<h2>Classification-Credit Score</h2>

Problem Statement:

You are working as a data scientist in a global finance company. Over the years, the company has collected basic bank details and gathered a lot of 
credit-related information. The management wants to build an intelligent system to segregate the people into credit score brackets to reduce the manual efforts. Given a person’s credit-related information, build a machine learning model that can classify the credit score.

Dataset Information:

Credit score dataset contains 1 lac records with 28 features.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
credit_score = pd.read_csv("C:/Users/AHSAN RAZA/Desktop/credit_score.csv")
print(credit_score)

In [ ]:
credit_score = credit_score.drop(columns=['ID','Customer_ID','Name','SSN','Type_of_Loan','Credit_History_Age'])
credit_score.head()

In [ ]:
credit_score.shape

In [ ]:
credit_score.info()

In [ ]:
credit_score.describe()

In [ ]:
credit_score.isnull().sum()

In [ ]:
credit_score['Age'] = credit_score['Age'].replace('_','')
credit_score['Age'] = credit_score['Age'].astype(int)
credit_score['Occupation'] = credit_score['Occupation'].replace('_______',np.nan)
credit_score['Annual_Income'] = credit_score['Annual_Income'].replace('_','')
credit_score['Annual_Income'] = credit_score['Annual_Income'].astype(float)
credit_score['Num_of_Loan'] = credit_score['Num_of_Loan'].replace('_','')
credit_score['Num_of_Loan'] = credit_score['Num_of_Loan'].astype(int)
credit_score['Num_of_Delayed_Payment'] = credit_score['Num_of_Delayed_Payment'].str.replace('_','')
credit_score['Num_of_Delayed_Payment'] = credit_score['Num_of_Delayed_Payment'].astype(float)
credit_score['Changed_Credit_Limit'] = credit_score['Changed_Credit_Limit'].replace('_',np.nan)
credit_score['Changed_Credit_Limit'] = credit_score['Changed_Credit_Limit'].astype(float)
credit_score['Credit_Mix'] = credit_score['Credit_Mix'].replace('_',np.nan)
credit_score['Credit_Mix'] = credit_score['Credit_Mix'].replace(['Standard','Good','Bad'],[1,2,0])
credit_score['Outstanding_Debt'] = credit_score['Outstanding_Debt'].str.replace('_','')
credit_score['Outstanding_Debt'] = credit_score['Outstanding_Debt'].astype(float)
credit_score['Payment_of_Min_Amount'] = credit_score['Payment_of_Min_Amount'].replace('NM','No')
credit_score['Payment_of_Min_Amount'] = credit_score['Payment_of_Min_Amount'].replace(['Yes','No'],[1,0])
credit_score['Amount_invested_monthly'] = credit_score['Amount_invested_monthly'].str.replace('_','')
credit_score['Amount_invested_monthly'] = credit_score['Amount_invested_monthly'].astype(float)
credit_score['Payment_Behaviour'] = credit_score['Payment_Behaviour'].replace('!@9#%8',np.nan)
credit_score['Monthly_Balance'] = credit_score['Monthly_Balance'].str.replace('_','')
credit_score['Monthly_Balance'] = credit_score['Monthly_Balance'].astype(float)
credit_score['Credit_Score'] = credit_score['Credit_Score'].replace(['Standard','Good','Poor'],[1,2,0])

In [ ]:
credit_score.info()

In [ ]:
credit_score.isnull().sum()

In [ ]:
credit_score = credit_score.fillna(method='ffill')
credit_score = credit_score.fillna(method='bfill')

In [ ]:
credit_score.isnull().sum()

In [ ]:
# Removing outliers from Age since all other columns values are relevant

In [ ]:
sns.boxplot(credit_score['Age'])
plt.xlabel('Age')
plt.ylabel('Count')
plt.show()

In [ ]:
col_names = credit_score['Age']
Q1 = credit_score.Age.quantile(0.25)
Q3 = credit_score.Age.quantile(0.75)
IQR = Q3-Q1
data = credit_score[(credit_score.Age >= Q1 - 1.5*IQR) & (credit_score.Age <= Q3 + 1.5*IQR)]
sns.boxplot(data['Age'])
plt.xlabel('Age')
plt.ylabel('Count')
plt.show()

In [ ]:
# Performing One Hot Encoding for categorical features of a dataframe

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
credit_score['Month'] = le.fit_transform(credit_score['Month'])
credit_score['Occupation'] = le.fit_transform(credit_score['Occupation'])
credit_score['Payment_Behaviour'] = le.fit_transform(credit_score['Payment_Behaviour'])

credit_score.info()

In [ ]:
# Selecting the features usin VIF. VIF should be less than 5. Here, all features have VIF values less than 5, so we will select all the features.

from statsmodels.stats.outliers_influence import variance_inflation_factor

col_list = []
for col in credit_score.columns:
    if((credit_score[col].dtype != 'object') & (col != 'Credit_Score')):
        col_list.append(col)

X = credit_score[col_list]
vif_data = pd.DataFrame()
vif_data['feature'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i)
                   for i in range(len(X.columns))]
print(vif_data)

In [ ]:
# Logistic Regression model

x = credit_score.drop(columns=['Credit_Score'])
y = credit_score['Credit_Score']

from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)

from sklearn.preprocessing import StandardScaler

sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()
lr.fit(x_train, y_train)
y_pred = lr.predict(x_test)

from sklearn.metrics import accuracy_score

print('Accuracy Score:',accuracy_score(y_test, y_pred))

pd.DataFrame({'Actual Value':y_test, 'Predited Value':y_pred})

In [ ]:
# Decision Tree model

from sklearn.tree import DecisionTreeClassifier

dtc = DecisionTreeClassifier()
dtc.fit(x_train, y_train)
y_pred = dtc.predict(x_test)

print('Accuracy Score:', accuracy_score(y_test, y_pred))

pd.DataFrame({'Actual Value':y_test, 'Predited Value':y_pred})

In [ ]:
# Hyperparameter Tuning on Decision Tree

from sklearn.model_selection import GridSearchCV

parameters = {'max_features':['log2','sqrt','auto'],
              'criterion':['entropy','gini'],
              'max_depth':[2,3,5,10,50],
              'min_samples_split':[2,3,50,100],
              'min_samples_leaf':[1,5,8,10]
             }
grid_obj = GridSearchCV(dtc, parameters)
grid_obj = grid_obj.fit(x_train,y_train)
dtc = grid_obj.best_estimator_
dtc.fit(x_train, y_train)
y_pred = dtc.predict(x_test)
acc_dtc = round(accuracy_score(y_test,y_pred)*100, 2)
print('Accuracy of Decision Tree model:', acc_dtc)

In [ ]:
# Random Forest model

from sklearn.ensemble import RandomForestClassifier

rfc = RandomForestClassifier()
rfc.fit(x_train, y_train)
y_pred = rfc.predict(x_test)

print('Accuracy Score:', accuracy_score(y_test,y_pred))

pd.DataFrame({'Actual Value':y_test, 'Predited Value':y_pred})